<h2> Random Forest Model: Surgical Data

Guiding questions: Can perioperative and intraoperative variables like existing diabetes and hypertension impact surgical outcomes like death, length of ICU stays, and general prolonged hospital stay?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('vitaldb_cleaned.csv')
df

,case_id,patient_id,admission_time_from_case_start_sec,discharge_time_from_case_start_sec,icu_days,in_hospital_death,age,sex,height,weight,...,intraoperative_midazolam,intraoperative_fentanyl,hospital_length_of_stay,had_icu_stay,prolonged_hospital_stay,age_numeric,emergency_operation_label,in_hospital_death_label,icu_stay_label,prolonged_hospital_stay_label
0,1,5955,-236220,627780,0,0,77,M,160.2,67.50,...,0.0,100,10.0,0,0,77.0,Non-emergency,Survived,No ICU stay,Not prolonged
1,2,2487,-221160,1506840,0,0,54,M,167.3,54.80,...,0.0,0,20.0,0,1,54.0,Non-emergency,Survived,No ICU stay,Prolonged stay
2,3,2861,-218640,40560,0,0,62,M,169.1,69.70,...,0.0,0,3.0,0,0,62.0,Non-emergency,Survived,No ICU stay,Not prolonged
3,4,1903,-201120,576480,1,0,74,M,160.6,53.00,...,0.0,100,9.0,1,0,74.0,Non-emergency,Survived,Had ICU stay,Not prolonged
4,5,4416,-67560,3734040,13,0,66,M,171.0,59.70,...,0.0,0,44.0,1,1,66.0,Emergency,Survived,Had ICU stay,Prolonged stay
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6383,6384,5583,-215340,648660,0,0,64,M,161.5,63.00,...,0.0,0,10.0,0,0,64.0,Non-emergency,Survived,No ICU stay,Not prolonged
6384,6385,2278,-225600,1675200,0,0,69,M,159.3,62.30,...,0.0,0,22.0,0,1,69.0,Non-emergency,Survived,No ICU stay,Prolonged stay
6385,6386,4045,-200460,836340,0,0,61,F,151.7,43.25,...,0.0,0,12.0,0,1,61.0,Non-emergency,Survived,No ICU stay,Prolonged stay
6386,6387,5230,-227760,377040,0,0,24,F,155.7,55.50,...,0.0,0,7.0,0,0,24.0,Non-emergency,Survived,No ICU stay,Not prolonged


In [2]:
df.columns

Index(['case_id', 'patient_id', 'admission_time_from_case_start_sec',
       'discharge_time_from_case_start_sec', 'icu_days', 'in_hospital_death',
       'age', 'sex', 'height', 'weight', 'bmi', 'asa_class',
       'emergency_operation', 'department', 'operation_type', 'diagnosis',
       'operation_name', 'surgical_approach', 'patient_position',
       'anesthesia_type', 'preoperative_hypertension', 'preoperative_diabetes',
       'preoperative_hemoglobin', 'preoperative_platelet_count',
       'preoperative_creatinine', 'intraoperative_estimated_blood_loss',
       'intraoperative_urine_output',
       'intraoperative_red_blood_cell_transfusion',
       'intraoperative_fresh_frozen_plasma_transfusion',
       'intraoperative_crystalloid', 'intraoperative_colloid',
       'intraoperative_propofol', 'intraoperative_midazolam',
       'intraoperative_fentanyl', 'hospital_length_of_stay', 'had_icu_stay',
       'prolonged_hospital_stay', 'age_numeric', 'emergency_operation_label',
     

In [3]:
df = df.drop(columns = ['emergency_operation_label', 'in_hospital_death_label', 'icu_stay_label','prolonged_hospital_stay_label', 'age'])
#these are the target variables. target variables need to be numeric so I can use scikit learn 
#also, if I keep it in the X matrix, these columns pose the possibility of multicollinearity because these are just relabelled versions of existing columns


In [4]:
df

,case_id,patient_id,admission_time_from_case_start_sec,discharge_time_from_case_start_sec,icu_days,in_hospital_death,sex,height,weight,bmi,...,intraoperative_fresh_frozen_plasma_transfusion,intraoperative_crystalloid,intraoperative_colloid,intraoperative_propofol,intraoperative_midazolam,intraoperative_fentanyl,hospital_length_of_stay,had_icu_stay,prolonged_hospital_stay,age_numeric
0,1,5955,-236220,627780,0,0,M,160.2,67.50,26.3,...,0,350.0,0,120,0.0,100,10.0,0,0,77.0
1,2,2487,-221160,1506840,0,0,M,167.3,54.80,19.6,...,0,800.0,0,150,0.0,0,20.0,0,1,54.0
2,3,2861,-218640,40560,0,0,M,169.1,69.70,24.4,...,0,200.0,0,0,0.0,0,3.0,0,0,62.0
3,4,1903,-201120,576480,1,0,M,160.6,53.00,20.5,...,0,2700.0,0,80,0.0,100,9.0,1,0,74.0
4,5,4416,-67560,3734040,13,0,M,171.0,59.70,20.4,...,8,7100.0,0,0,0.0,0,44.0,1,1,66.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6383,6384,5583,-215340,648660,0,0,M,161.5,63.00,24.2,...,0,550.0,0,150,0.0,0,10.0,0,0,64.0
6384,6385,2278,-225600,1675200,0,0,M,159.3,62.30,24.6,...,0,2500.0,0,100,0.0,0,22.0,0,1,69.0
6385,6386,4045,-200460,836340,0,0,F,151.7,43.25,18.8,...,0,1300.0,0,70,0.0,0,12.0,0,1,61.0
6386,6387,5230,-227760,377040,0,0,F,155.7,55.50,22.9,...,0,1150.0,0,120,0.0,0,7.0,0,0,24.0


<h3> In-Hospital Deaths

Random Forest Model

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

In [6]:
#preparing training and test sets
X1 = df.drop(columns = 'in_hospital_death')
X_death = pd.get_dummies(X1, drop_first=True) #one hot encoding
y_death = df['in_hospital_death']

X_train_death, X_test_death, y_train_death, y_test_death = train_test_split(X_death, y_death, random_state=1)

In [7]:
rf_death = RandomForestClassifier(max_features = 0.5, 
                                  max_depth= 20, class_weight='balanced')
rf_death.fit(X_train_death, y_train_death)
rf_pred_death_train = rf_death.predict(X_train_death)
rf_pred_death_test = rf_death.predict(X_test_death)


rf_death_metrics_train = [accuracy_score(y_train_death, rf_pred_death_train), precision_score(y_train_death, rf_pred_death_train), 
                    recall_score(y_train_death, rf_pred_death_train), f1_score(y_train_death, rf_pred_death_train)]

rf_death_metrics_test = [accuracy_score(y_test_death, rf_pred_death_test), precision_score(y_test_death, rf_pred_death_test), 
                    recall_score(y_test_death, rf_pred_death_test), f1_score(y_test_death, rf_pred_death_test)]

rf_matrix_death = [rf_death_metrics_train, rf_death_metrics_test]

rf_results_death = pd.DataFrame(rf_matrix_death, index= ['train', 'test'], columns=['Accuracy', 'Precision', 'Recall', 'F1 Score'])
rf_results_death

,Accuracy,Precision,Recall,F1 Score
train,0.999791,0.977273,1.000000,0.988506
test,0.992486,0.666667,0.285714,0.400000


In [8]:
from xgboost import XGBClassifier


xgb_death = XGBClassifier(random_state=42)
xgb_death.fit(X_train_death, y_train_death, verbose=False)
xgb_death_preds_train = xgb_death.predict(X_train_death)
xgb_death_preds_test = xgb_death.predict(X_test_death)


xgb_death_metrics_train = [accuracy_score(y_train_death, xgb_death_preds_train), precision_score(y_train_death, xgb_death_preds_train), 
                    recall_score(y_train_death, xgb_death_preds_train), f1_score(y_train_death, xgb_death_preds_train)]

xgb_death_metrics_test = [accuracy_score(y_test_death, xgb_death_preds_test), precision_score(y_test_death, xgb_death_preds_test), 
                    recall_score(y_test_death, xgb_death_preds_test), f1_score(y_test_death, xgb_death_preds_test)]

xgb_matrix = [xgb_death_metrics_train, xgb_death_metrics_test]

xgb_results_death = pd.DataFrame(xgb_matrix, index= ['train', 'test'], columns=['Accuracy', 'Precision', 'Recall', 'F1 Score'])

xgb_results_death


,Accuracy,Precision,Recall,F1 Score
train,1.000000,1.000000,1.000000,1.0
test,0.993738,0.833333,0.357143,0.5


In [9]:
rf_feat_importance = pd.DataFrame({'Importance': rf_death.feature_importances_}, index = X_death.columns).sort_values('Importance', ascending=False)
rf_feat_importance = rf_feat_importance[rf_feat_importance['Importance'] > 0.001].sort_values('Importance', ascending=False)
rf_feat_importance
#removing columns that have a feature importance of less than 0.001

,Importance
hospital_length_of_stay,0.113133
discharge_time_from_case_start_sec,0.110168
case_id,0.080362
icu_days,0.065602
preoperative_creatinine,0.060383
...,...
operation_name_Sleeve lobectomy,0.001355
operation_type_Major resection,0.001301
intraoperative_fentanyl,0.001224
diagnosis_Advanced gastric cancer,0.001181


In [10]:
cols_for_xg = list(rf_feat_importance.index)
#let's just put the columns that have feature importance in xgboost

In [11]:
'''
This is some code I wrote for an XGBoost model that might not be good. But keeping this for later reference.

X_train_death_feat_imp = X_train_death[cols_for_xg] 
X_test_death_feat_imp = X_test_death[cols_for_xg]

the two lines above are the faulty lines. but this might be good intuition for something else. 

xgb_death = XGBClassifier(random_state=42)
xgb_death.fit(X_train_death_feat_imp, y_train_death, verbose=False)
xgb_death_preds_train = xgb_death.predict(X_train_death_feat_imp)
xgb_death_preds_test = xgb_death.predict(X_test_death_feat_imp)


xgb_death_metrics_train = [accuracy_score(y_train_death, xgb_death_preds_train), precision_score(y_train_death, xgb_death_preds_train), 
                    recall_score(y_train_death, xgb_death_preds_train), f1_score(y_train_death, xgb_death_preds_train)]

xgb_death_metrics_test = [accuracy_score(y_test_death, xgb_death_preds_test), precision_score(y_test_death, xgb_death_preds_test), 
                    recall_score(y_test_death, xgb_death_preds_test), f1_score(y_test_death, xgb_death_preds_test)]

xgb_matrix = [xgb_death_metrics_train, xgb_death_metrics_test]

xgb_results = pd.DataFrame(xgb_matrix, index= ['train', 'test'], columns=['Accuracy', 'Precision', 'Recall', 'F1 Score'])

xgb_results

'''

"\nThis is some code I wrote for an XGBoost model that might not be good. But keeping this for later reference.\n\nX_train_death_feat_imp = X_train_death[cols_for_xg] \nX_test_death_feat_imp = X_test_death[cols_for_xg]\n\nthe two lines above are the faulty lines. but this might be good intuition for something else. \n\nxgb_death = XGBClassifier(random_state=42)\nxgb_death.fit(X_train_death_feat_imp, y_train_death, verbose=False)\nxgb_death_preds_train = xgb_death.predict(X_train_death_feat_imp)\nxgb_death_preds_test = xgb_death.predict(X_test_death_feat_imp)\n\n\nxgb_death_metrics_train = [accuracy_score(y_train_death, xgb_death_preds_train), precision_score(y_train_death, xgb_death_preds_train), \n                    recall_score(y_train_death, xgb_death_preds_train), f1_score(y_train_death, xgb_death_preds_train)]\n\nxgb_death_metrics_test = [accuracy_score(y_test_death, xgb_death_preds_test), precision_score(y_test_death, xgb_death_preds_test), \n                    recall_score(y_t

<h2> Prolonged Hospital Stay

Splitting Training and Test Sets

In [24]:
X2 = df.drop(columns = 'prolonged_hospital_stay') #let's shorten prolonged hospital stay to phs
X_phs = pd.get_dummies(X2, drop_first=True) #one hot encoding
y_phs = df['prolonged_hospital_stay']

X_train_phs, X_test_phs, y_train_phs, y_test_phs = train_test_split(X_phs, y_phs, random_state=1)

Random Forest Model 

In [25]:
rf_phs = RandomForestClassifier(random_state = 42, class_weight = 'balanced', max_depth = 7, n_estimators=500, min_samples_split=5)
rf_phs.fit(X_train_phs, y_train_phs)
rf_pred_phs_train = rf_phs.predict(X_train_phs)
rf_pred_phs_test = rf_phs.predict(X_test_phs)


rf_phs_metrics_train = [accuracy_score(y_train_phs, rf_pred_phs_train), precision_score(y_train_phs, rf_pred_phs_train), 
                    recall_score(y_train_phs, rf_pred_phs_train), f1_score(y_train_phs, rf_pred_phs_train)]

rf_phs_metrics_test = [accuracy_score(y_test_phs, rf_pred_phs_test), precision_score(y_test_phs, rf_pred_phs_test), 
                    recall_score(y_test_phs, rf_pred_phs_test), f1_score(y_test_phs, rf_pred_phs_test)]

rf_matrix_phs = [rf_phs_metrics_train, rf_phs_metrics_test]

rf_results_phs = pd.DataFrame(rf_matrix_phs, index= ['train', 'test'], columns=['Accuracy', 'Precision', 'Recall', 'F1 Score'])
rf_results_phs

,Accuracy,Precision,Recall,F1 Score
train,0.992277,0.972642,0.992300,0.982373
test,0.984972,0.954545,0.972222,0.963303


Random Forest Model: Feature Importance

In [14]:
rf_phs_feat_importance = pd.DataFrame({'Importance': rf_phs.feature_importances_}, index = X_phs.columns).sort_values('Importance', ascending=False)
rf_phs_feat_importance = rf_phs_feat_importance[rf_phs_feat_importance['Importance'] > 0.001].sort_values('Importance', ascending=False).head(30)
rf_phs_feat_importance.head(5)

,Importance
hospital_length_of_stay,0.266211
discharge_time_from_case_start_sec,0.211565
admission_time_from_case_start_sec,0.062331
intraoperative_crystalloid,0.031019
icu_days,0.026728


In [15]:
cols_for_xg_phs = list(rf_phs_feat_importance.index)
cols_for_xg_phs

['hospital_length_of_stay',
 'discharge_time_from_case_start_sec',
 'admission_time_from_case_start_sec',
 'intraoperative_crystalloid',
 'icu_days',
 'intraoperative_estimated_blood_loss',
 'preoperative_hemoglobin',
 'intraoperative_urine_output',
 'had_icu_stay',
 'asa_class',
 'weight',
 'preoperative_creatinine',
 'bmi',
 'preoperative_platelet_count',
 'intraoperative_colloid',
 'height',
 'age_numeric',
 'patient_id',
 'case_id',
 'operation_name_Kidney transplantation',
 'surgical_approach_Videoscopic',
 'operation_type_Others',
 'intraoperative_red_blood_cell_transfusion',
 'diagnosis_End stage renal disease',
 'emergency_operation',
 'intraoperative_propofol',
 'operation_name_Liver transplantation',
 'patient_position_Supine',
 'intraoperative_fresh_frozen_plasma_transfusion',
 'operation_type_Transplantation']

XGBoost Model

In [16]:
from sklearn.model_selection import GridSearchCV

#(columns = ['hospital_length_of_stay', 'discharge_time_from_case_start_sec'])

param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'min_child_weight': [1, 5, 10],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

neg = (y_train_phs == 0).sum()
pos = (y_train_phs == 1).sum()

xgb = XGBClassifier(n_estimators=300,
    scale_pos_weight=neg/pos,
    random_state=42)

grid = GridSearchCV(
    xgb,
    param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

grid.fit(X_train_phs, y_train_phs, verbose=False)

print(grid.best_params_)
print(grid.best_score_)


KeyboardInterrupt: 

In [26]:
X_train_phs_feat_imp = X_train_phs[cols_for_xg_phs].drop(columns = 'hospital_length_of_stay')
X_test_phs_feat_imp = X_test_phs[cols_for_xg_phs].drop(columns = 'hospital_length_of_stay')

xgb_phs = XGBClassifier(n_estimators=300,
    scale_pos_weight=neg/pos,
    random_state=42,
    colsample_bytree =  0.8,
    learning_rate = 0.01,
    max_depth = 6,
    min_child_weight = 1,
    subsample = 0.8
    )

xgb_phs.fit(X_train_phs_feat_imp, y_train_phs, verbose=False)
xgb_phs_preds_train = xgb_phs.predict(X_train_phs_feat_imp)
xgb_phs_preds_test = xgb_phs.predict(X_test_phs_feat_imp)


xgb_phs_metrics_train = [accuracy_score(y_train_phs, xgb_phs_preds_train), precision_score(y_train_phs, xgb_phs_preds_train), 
                    recall_score(y_train_phs, xgb_phs_preds_train), f1_score(y_train_phs, xgb_phs_preds_train)]

xgb_phs_metrics_test = [accuracy_score(y_test_phs, xgb_phs_preds_test), precision_score(y_test_phs, xgb_phs_preds_test), 
                    recall_score(y_test_phs, xgb_phs_preds_test), f1_score(y_test_phs, xgb_phs_preds_test)]

xgb_matrix_phs = [xgb_phs_metrics_train, xgb_phs_metrics_test]

xgb_results_phs = pd.DataFrame(xgb_matrix_phs, index= ['train', 'test'], columns=['Accuracy', 'Precision', 'Recall', 'F1 Score'])

xgb_results_phs


,Accuracy,Precision,Recall,F1 Score
train,0.999583,0.998079,1.000000,0.999038
test,0.998121,0.993846,0.996914,0.995378


In [18]:
X_train_phs[['hospital_length_of_stay', 'discharge_time_from_case_start_sec']].corr()

,hospital_length_of_stay,discharge_time_from_case_start_sec
hospital_length_of_stay,1.000000,0.029773
discharge_time_from_case_start_sec,0.029773,1.000000


In [19]:
X_train_phs_feat_imp.columns

Index(['discharge_time_from_case_start_sec',
       'admission_time_from_case_start_sec', 'intraoperative_crystalloid',
       'icu_days', 'intraoperative_estimated_blood_loss',
       'preoperative_hemoglobin', 'intraoperative_urine_output',
       'had_icu_stay', 'asa_class', 'weight', 'preoperative_creatinine', 'bmi',
       'preoperative_platelet_count', 'intraoperative_colloid', 'height',
       'age_numeric', 'patient_id', 'case_id',
       'operation_name_Kidney transplantation',
       'surgical_approach_Videoscopic', 'operation_type_Others',
       'intraoperative_red_blood_cell_transfusion',
       'diagnosis_End stage renal disease', 'emergency_operation',
       'intraoperative_propofol', 'operation_name_Liver transplantation',
       'patient_position_Supine',
       'intraoperative_fresh_frozen_plasma_transfusion',
       'operation_type_Transplantation'],
      dtype='object')

In [20]:
df['had_icu_stay'].value_counts()

had_icu_stay
0    5184
1    1204
Name: count, dtype: int64

<h2> ICU Stays

Splitting Training and Test Set

In [28]:
X3 = df.drop(columns = ['had_icu_stay', 'icu_days']	) #let's shorten prolonged hospital stay to phs
X_icu = pd.get_dummies(X3, drop_first=True) #one hot encoding
y_icu = df['had_icu_stay']

X_train_icu, X_test_icu, y_train_icu, y_test_icu = train_test_split(X_icu, y_icu, random_state=1, stratify=y_icu)

Random Forest Model

In [ ]:
rf_icu = RandomForestClassifier(random_state = 42, class_weight = 'balanced', max_depth = 21, n_estimators=500, min_samples_split=5)
rf_icu.fit(X_train_icu, y_train_icu)
rf_pred_icu_train = rf_icu.predict(X_train_icu)
rf_pred_icu_test = rf_icu.predict(X_test_icu)


rf_icu_metrics_train = [accuracy_score(y_train_icu, rf_pred_icu_train), precision_score(y_train_icu, rf_pred_icu_train), 
                    recall_score(y_train_icu, rf_pred_icu_train), f1_score(y_train_icu, rf_pred_icu_train)]

rf_icu_metrics_test = [accuracy_score(y_test_icu, rf_pred_icu_test), precision_score(y_test_icu, rf_pred_icu_test), 
                    recall_score(y_test_icu, rf_pred_icu_test), f1_score(y_test_icu, rf_pred_icu_test)]

rf_matrix_icu = [rf_icu_metrics_train, rf_icu_metrics_test]

rf_results_icu = pd.DataFrame(rf_matrix_icu, index= ['train', 'test'], columns=['Accuracy', 'Precision', 'Recall', 'F1 Score'])
rf_results_icu

,Accuracy,Precision,Recall,F1 Score
train,0.847631,0.560279,0.890365,0.687767
test,0.859111,0.582609,0.890365,0.704336


In [ ]:
#this helped with hyperparameter tuning and was also cross validation as well

'''param_grid1 = {
    'n_estimators': [100, 300, 500],
    'max_depth': [5, 10, 15],
    'min_samples_leaf': [1, 5, 10],
    'min_samples_split': [2, 10, 20],
    'class_weight': ['balanced']
}
rf_grid = GridSearchCV(
    rf_icu,
    param_grid1,
    scoring='f1',
    cv=5
)

rf_grid.fit(X_train_icu, y_train_icu)
print(rf_grid.best_params_)
print(rf_grid.best_score_)'''

{'class_weight': 'balanced', 'max_depth': 15, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 500}
0.7246243411488142


Feature Importance

In [ ]:
rf_icu_feat_importance = pd.DataFrame({'Importance': rf_icu.feature_importances_}, index = X_icu.columns).sort_values('Importance', ascending=False)
rf_icu_feat_importance = rf_icu_feat_importance[rf_icu_feat_importance['Importance'] > 0.001].sort_values('Importance', ascending=False).head(30)
rf_icu_feat_importance

,Importance
discharge_time_from_case_start_sec,0.093458
hospital_length_of_stay,0.070606
operation_name_Lung lobectomy,0.064496
prolonged_hospital_stay,0.053140
department_Thoracic surgery,0.044477
intraoperative_crystalloid,0.043510
intraoperative_estimated_blood_loss,0.034104
operation_type_Major resection,0.033113
intraoperative_urine_output,0.029004
asa_class,0.028158


In [ ]:
cols_for_xg_icu = list(rf_icu_feat_importance.index)

XGBoost Model 

In [ ]:


'''neg = (y_train_icu == 0).sum()
pos = (y_train_icu == 1).sum()

xgb2 = XGBClassifier(n_estimators=300,
    scale_pos_weight=neg/pos,
    random_state=42)

grid = GridSearchCV(
    xgb,
    param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

grid.fit(X_train_icu, y_train_icu, verbose=False)

print(grid.best_params_)
print(grid.best_score_)'''



{'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 7, 'min_child_weight': 5, 'subsample': 1.0}
0.7336030806038281


In [ ]:
X_train_icu_feat_imp = X_train_icu[cols_for_xg_icu]
X_test_icu_feat_imp = X_test_icu[cols_for_xg_icu]

xgb_icu = XGBClassifier(n_estimators=300,
    scale_pos_weight=neg/pos,
    random_state=42,
    colsample_bytree =  0.8,
    learning_rate = 0.05,
    max_depth = 7,
    min_child_weight = 5,
    subsample = 1.0
    )

xgb_icu.fit(X_train_icu_feat_imp, y_train_icu, verbose=False)
xgb_icu_preds_train = xgb_icu.predict(X_train_icu_feat_imp)
xgb_icu_preds_test = xgb_icu.predict(X_test_icu_feat_imp)


xgb_icu_metrics_train = [accuracy_score(y_train_icu, xgb_icu_preds_train), precision_score(y_train_icu, xgb_icu_preds_train), 
                    recall_score(y_train_icu, xgb_icu_preds_train), f1_score(y_train_icu, xgb_icu_preds_train)]

xgb_icu_metrics_test = [accuracy_score(y_test_icu, xgb_icu_preds_test), precision_score(y_test_icu, xgb_icu_preds_test), 
                    recall_score(y_test_icu, xgb_icu_preds_test), f1_score(y_test_icu, xgb_icu_preds_test)]

xgb_matrix_icu = [xgb_icu_metrics_train, xgb_icu_metrics_test]

xgb_results_icu = pd.DataFrame(xgb_matrix_icu, index= ['train', 'test'], columns=['Accuracy', 'Precision', 'Recall', 'F1 Score'])

xgb_results_icu


,Accuracy,Precision,Recall,F1 Score
train,0.985598,0.929012,1.000000,0.963200
test,0.906074,0.717579,0.827243,0.768519
